# Lab: 2D Convolution — Multi-Channel AXI-Stream & AXI-Lite

This notebook interfaces with the generalized HLS 2D Convolution IP (`conv2d_stream`).
It supports:
- **Multiple input channels (Cin)** and **output channels (Cout)**
- **AXI-Stream** via DMA for pixel data (interleaved channel order)
- **AXI-Lite** for scalar registers: `rows`, `cols`, `Cin`, `Cout`
- **AXI-Lite Memory Map** for the 3D kernel array `[MAX_COUT][MAX_CIN][K*K]`

### HLS Output Pixel Format
The HLS kernel writes output pixels interleaved as:
```
(row=0,col=0,oc=0), (row=0,col=0,oc=1), ..., (row=R-1,col=C-1,oc=Cout-1)
```
Total output words = `rows * cols * Cout`

### HLS Input Pixel Format
The HLS kernel reads pixels from the AXI-Stream for each `(pi, pj)` position in padded space,
but only reads `Cin` packets when the position is valid (non-padded). So input must be sent as:
```
(row=0,col=0,ic=0), (row=0,col=0,ic=1), ..., (row=R-1,col=C-1,ic=Cin-1)
```
Total input words = `rows * cols * Cin`

## 1. Load Overlay

In [ ]:
import time
import struct
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate
from scipy.signal import convolve2d

ol = Overlay("design_1_float.bit")
print("Overlay loaded")
print("IP blocks:", list(ol.ip_dict.keys()))

dma    = ol.axi_dma_0
conv2d = ol.conv2d_stream_1

## 2. Configuration Parameters

Set image dimensions and channel counts here.
- `MAX_WIDTH = 128` in HLS → `COLS` must be ≤ 128
- `MAX_CIN = MAX_COUT = 8` in HLS → channel counts must be ≤ 8
- Kernel is always 3×3 (`K=3` in HLS)

In [ ]:
# ── Image & channel dimensions ───────────────────────────────────
ROWS = 64
COLS = 64
CIN  = 3   # Input channels  (must be ≤ MAX_CIN=8)
COUT = 2   # Output channels (must be ≤ MAX_COUT=8)

# ── HLS hardware constants (must match 2DCONV_stream.h) ──────────
K        = 3
MAX_CIN  = 8
MAX_COUT = 8

assert COLS <= 128,  f"COLS={COLS} exceeds MAX_WIDTH=128"
assert CIN  <= MAX_CIN,  f"CIN={CIN} exceeds MAX_CIN={MAX_CIN}"
assert COUT <= MAX_COUT, f"COUT={COUT} exceeds MAX_COUT={MAX_COUT}"

print(f"Config: {ROWS}x{COLS} image, Cin={CIN}, Cout={COUT}, Kernel={K}x{K}")

## 3. Prepare Test Image & Kernels

Create a synthetic multi-channel image and a kernel set of shape `[COUT, CIN, K*K]`.

Each output channel `oc` accumulates over all input channels `ic`:
```
out[oc] = sum over ic of (in[ic] convolved with kernel[oc][ic])
```

In [ ]:
# ── Synthetic multi-channel input image ──────────────────────────
# Shape: (CIN, ROWS, COLS) — each channel has a different pattern
image_in = np.zeros((CIN, ROWS, COLS), dtype=np.float32)
for ic in range(CIN):
    # Offset block per channel so we can distinguish outputs
    offset = ic * 5
    image_in[ic, 20+offset:36+offset, 20:36] = 1.0
    image_in[ic] = np.clip(image_in[ic], 0, 1)  # stay in [0,1]

# ── Kernel set: shape (COUT, CIN, K*K) ───────────────────────────
# We will zero-pad to (MAX_COUT, MAX_CIN, K*K) when writing to HW
kernels = np.zeros((COUT, CIN, K*K), dtype=np.float32)

# Output channel 0: Edge detection applied to all input channels equally
edge_kernel = np.array([-1,-1,-1, -1,8,-1, -1,-1,-1], dtype=np.float32)
for ic in range(CIN):
    kernels[0, ic] = edge_kernel / CIN  # normalize by Cin

# Output channel 1: Box blur / average filter on all input channels
blur_kernel = np.ones(K*K, dtype=np.float32) / (K*K)
for ic in range(CIN):
    kernels[1, ic] = blur_kernel / CIN  # normalize by Cin

print(f"image_in  shape: {image_in.shape}  (CIN x ROWS x COLS)")
print(f"kernels   shape: {kernels.shape}   (COUT x CIN x K*K)")
print(f"\nKernel[0] (edge, ic=0):\n{kernels[0,0].reshape(K,K)}")
print(f"\nKernel[1] (blur, ic=0):\n{kernels[1,0].reshape(K,K)}")

## 4. Software Golden Model

Compute expected output in Python using `scipy.convolve2d` to verify hardware output later.

In [ ]:
def sw_conv2d(image_in, kernels, COUT, CIN, K):
    """Software reference: matches HLS accumulation over all Cin channels."""
    ROWS, COLS = image_in.shape[1], image_in.shape[2]
    out = np.zeros((COUT, ROWS, COLS), dtype=np.float32)
    for oc in range(COUT):
        for ic in range(CIN):
            k2d = kernels[oc, ic].reshape(K, K)
            # scipy convolve2d with 'same' matches HLS window-based approach
            out[oc] += convolve2d(image_in[ic], k2d,
                                  mode='same', boundary='fill', fillvalue=0)
    return out

image_out_sw = sw_conv2d(image_in, kernels, COUT, CIN, K)
print(f"SW output shape: {image_out_sw.shape}  (COUT x ROWS x COLS)")

## 5. Configure IP via AXI-Lite

Register map (verify offsets against your `.hwh` / `xconv2d_stream_hw.h`):

| Register | Offset | Description |
|----------|--------|-------------|
| `ap_ctrl` | `0x00` | Control (ap_start bit 0) |
| `rows`    | `0x10` | Number of rows |
| `cols`    | `0x18` | Number of cols |
| `Cin`     | `0x20` | Input channels |
| `Cout`    | `0x28` | Output channels |
| `kernel`  | `0x40` | Base of kernel array — `MAX_COUT × MAX_CIN × K*K` floats |

> **Important:** Kernel is stored in HW as `kernel[MAX_COUT][MAX_CIN][K*K]`.
> Strides: `oc_stride = MAX_CIN * K*K * 4`, `ic_stride = K*K * 4`

In [ ]:
# ── AXI-Lite register offsets ────────────────────────────────────
# Verify these against xconv2d_stream_hw.h from your HLS export!
CTRL_REG    = 0x00
ROWS_OFFSET = 0x10
COLS_OFFSET = 0x18
CIN_OFFSET  = 0x20
COUT_OFFSET = 0x28
KERNEL_BASE = 0x40   # Base address of kernel[MAX_COUT][MAX_CIN][K*K]

AP_START = 0x01
AP_DONE  = 0x02
AP_IDLE  = 0x04

def float_to_uint(f):
    """Reinterpret float32 bits as uint32 for AXI-Lite write."""
    return struct.unpack('<I', struct.pack('<f', float(f)))[0]

# ── Write scalar registers ────────────────────────────────────────
conv2d.write(ROWS_OFFSET, ROWS)
conv2d.write(COLS_OFFSET, COLS)
conv2d.write(CIN_OFFSET,  CIN)
conv2d.write(COUT_OFFSET, COUT)
print(f"Written: rows={ROWS}, cols={COLS}, Cin={CIN}, Cout={COUT}")

# ── Write 3D kernel to AXI-Lite memory map ───────────────────────
# HW layout: kernel[MAX_COUT][MAX_CIN][K*K]  (all indices in HW-space)
# Byte stride: each float = 4 bytes
# oc_stride = MAX_CIN * K*K * 4 bytes
# ic_stride = K*K * 4 bytes
oc_stride = MAX_CIN * K * K * 4
ic_stride = K * K * 4

for oc in range(COUT):
    for ic in range(CIN):
        for k in range(K * K):
            byte_offset = oc * oc_stride + ic * ic_stride + k * 4
            addr = KERNEL_BASE + byte_offset
            conv2d.write(addr, float_to_uint(kernels[oc, ic, k]))

print(f"Kernel written: {COUT}×{CIN}×{K*K} = {COUT*CIN*K*K} floats")
print(f"Register map spans: 0x{KERNEL_BASE:02X} – "
      f"0x{KERNEL_BASE + MAX_COUT*MAX_CIN*K*K*4 - 4:02X}")

## 6. Prepare DMA Buffers

### Input buffer layout
The HLS kernel reads `Cin` packets per valid pixel (row-major, channels last):
```
[ pix(0,0,ic0), pix(0,0,ic1), ..., pix(0,0,icN),
  pix(0,1,ic0), ...,
  pix(R-1,C-1,ic0), ..., pix(R-1,C-1,icN) ]
```
Total = `ROWS * COLS * CIN` words

### Output buffer layout
The HLS kernel writes `Cout` packets per valid output pixel:
```
[ out(0,0,oc0), out(0,0,oc1), ..., out(0,0,ocM),
  out(0,1,oc0), ...,
  out(R-1,C-1,oc0), ..., out(R-1,C-1,ocM) ]
```
Total = `ROWS * COLS * COUT` words

In [ ]:
# ── Pack input: (CIN, ROWS, COLS) → row-major, channels-last ─────
# Transpose from (CIN, ROWS, COLS) to (ROWS, COLS, CIN) then flatten
image_in_hwc = np.transpose(image_in, (1, 2, 0))  # shape: (ROWS, COLS, CIN)
in_flat = image_in_hwc.flatten().astype(np.float32)

assert len(in_flat) == ROWS * COLS * CIN
print(f"Input  buffer: {len(in_flat)} words  ({ROWS}×{COLS}×{CIN})")

# ── Allocate DMA buffers ──────────────────────────────────────────
in_buf  = allocate(shape=(ROWS * COLS * CIN,),  dtype=np.float32)
out_buf = allocate(shape=(ROWS * COLS * COUT,), dtype=np.float32)

np.copyto(in_buf, in_flat)
out_buf[:] = 0

print(f"Output buffer: {len(out_buf)} words  ({ROWS}×{COLS}×{COUT})")

## 7. Execute DMA Transfer

In [ ]:
# ── Safely (re)start DMA channels ────────────────────────────────
for ch in [dma.sendchannel, dma.recvchannel]:
    try:
        ch.stop()
    except Exception:
        pass
    ch.start()

# ── Assert IP is idle before starting ────────────────────────────
status = conv2d.read(CTRL_REG)
assert status & AP_IDLE, f"IP not idle before start! CTRL={status:#04x}"

# ── Start IP FIRST (must be ready before DMA pushes data) ────────
conv2d.write(CTRL_REG, AP_START)

# ── Start DMA: recv armed before send ────────────────────────────
t0 = time.perf_counter()

dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)

dma.sendchannel.wait()
dma.recvchannel.wait()

t_dma = time.perf_counter() - t0

# ── Verify IP completion ──────────────────────────────────────────
status = conv2d.read(CTRL_REG)
if status & AP_DONE:
    print("✅ IP ap_done asserted")
else:
    print(f"⚠️  ap_done NOT set. CTRL={status:#04x}")

print(f"Processed {ROWS}×{COLS} × Cin={CIN} → Cout={COUT} in {t_dma*1e3:.2f} ms")

## 8. Unpack Hardware Output

Reshape the flat DMA output `[ROWS*COLS*COUT]` back to `(COUT, ROWS, COLS)`.

In [ ]:
# out_buf is interleaved: (ROWS, COLS, COUT) → transpose to (COUT, ROWS, COLS)
out_hwc = np.array(out_buf).reshape((ROWS, COLS, COUT))  # channels last
image_out_hw = np.transpose(out_hwc, (2, 0, 1))          # → (COUT, ROWS, COLS)

print(f"HW output shape: {image_out_hw.shape}  (COUT x ROWS x COLS)")

## 9. Verification

Compare each output channel to the software golden model.
The outermost 1-pixel border may be invalid (HLS line-buffer fill delay), so we crop `[1:-1, 1:-1]`.

In [ ]:
print("=" * 50)
print("Verification (inner window [1:-1, 1:-1])")
print("=" * 50)

all_pass = True
for oc in range(COUT):
    hw_valid = image_out_hw[oc, 1:-1, 1:-1]
    sw_valid = image_out_sw[oc, 1:-1, 1:-1]
    max_diff = np.max(np.abs(hw_valid - sw_valid))
    status   = "PASS ✅" if max_diff < 1e-4 else "FAIL ❌"
    print(f"  Output channel {oc}: max|HW-SW| = {max_diff:.2e}  [{status}]")
    if max_diff >= 1e-4:
        all_pass = False

print()
print("Overall:", "PASS ✅" if all_pass else "FAIL ❌ — check kernel offsets or Cin/Cout register values")

## 10. Visualization

In [ ]:
fig, axes = plt.subplots(COUT, 3, figsize=(15, 5 * COUT))

# Handle the case where COUT=1 (axes won't be 2D)
if COUT == 1:
    axes = axes[np.newaxis, :]

for oc in range(COUT):
    # Show input channel 0 as reference
    axes[oc, 0].imshow(image_in[0], cmap='gray')
    axes[oc, 0].set_title(f"Input (ic=0)")

    axes[oc, 1].imshow(image_out_sw[oc], cmap='gray')
    axes[oc, 1].set_title(f"SW Output (oc={oc})")

    axes[oc, 2].imshow(image_out_hw[oc], cmap='gray')
    axes[oc, 2].set_title(f"HW Output (oc={oc})")

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 11. Cleanup

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("Buffers freed.")

---
## Appendix: Kernel Offset Calculator

Use this cell to compute and verify AXI-Lite addresses for any kernel element.

In [ ]:
def kernel_addr(oc, ic, k_idx, KERNEL_BASE=0x40, MAX_CIN=8, K=3):
    """Compute AXI-Lite byte address for kernel[oc][ic][k_idx]."""
    oc_stride = MAX_CIN * K * K * 4
    ic_stride = K * K * 4
    return KERNEL_BASE + oc * oc_stride + ic * ic_stride + k_idx * 4

# Example: print all addresses for a 2-out × 3-in × 9-tap kernel
print(f"{'oc':>4} {'ic':>4} {'k':>4}  {'addr':>8}")
print("-" * 28)
for oc in range(COUT):
    for ic in range(CIN):
        for k in range(K*K):
            addr = kernel_addr(oc, ic, k)
            print(f"{oc:>4} {ic:>4} {k:>4}  0x{addr:04X}")